In [10]:
import pandas as pd 
import ast
import numpy as np

In [11]:
%cd Thesis-FOS-BinaryClass-WSD/Ensemble Model/

[WinError 3] The system cannot find the path specified: 'Thesis-FOS-BinaryClass-WSD/Ensemble Model/'
c:\Users\Miguel\Documents\Project Source Files\IT Work\School\Thesis\Thesis-FOS-BinaryClass-WSD\Ensemble Model


c:\Users\Miguel\Documents\Project Source Files\IT Work\School\Thesis\.venv\Lib\site-packages\IPython\core\magics\osm.py:393: UserWarning: This is now an optional IPython functionality, using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})


# Load Models

## Sentence Transformer -- Sentence Embeddings


In [12]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('sentence-transformers/LaBSE')

# Processing

## Load Dataset

In [13]:
df_test = pd.read_excel('../../Dataset/Test_Set.xlsx')
df_train= pd.read_excel('../../Dataset/Train_Set.xlsx')
display(df_test.head(1),df_test.shape,df_train.head(1),df_train.shape)

,FOS,Example,Non-literal Word Sense,Literal Word Sense,Verb,Is FOS
0,hatod og patay,Si Juan hatod og patay sa iyang paryente nga n...,to attend a funeral or memorial service,to attend a funeral or memorial service,"['hatod', 'patay']",1


(32, 6)

,FOS,Example,Non-literal Word Sense,Literal Word Sense,Verb,Is FOS
0,lapas na sa kalendaryo,"Si Ana lapas na sa kalendaryo, pero dili gihap...",a person over the age of 31,a person over the age of 31,['lapas'],1


(124, 6)

## Sentence Transformer

Removing the label column from the dataset for clarity

In [14]:
df_train.drop(columns=['Is FOS'], inplace=True)
df = df_train.copy()

In [17]:
# Limit to 30 so that embeddings are retrieved faster
df['Example'] = df['Example'].apply(lambda x: x[:30],)
df.head(3)

,FOS,Example,Non-literal Word Sense,Literal Word Sense,Verb
0,lapas na sa kalendaryo,"Si Ana lapas na sa kalendaryo,",a person over the age of 31,a person over the age of 31,['lapas']
1,molayag sa kalipay,Ang mga magkauban nagmolayag s,(euphemistic) to have sexual intercourse,(euphemistic) to have sexual intercourse,['molayag']
2,bunal,Gidala niya ang bunal aron map,stick used in beating or whipping an animal or...,stick used in beating or whipping an animal or...,['bunal']


In [20]:
def GetEmbeddings(words:str) -> np.ndarray:
    return embedding_model.encode(words)

def ComputeSimilarity(embedding1,embedding2):
    return embedding_model.similarity(embedding1, embedding2)

### Generate Embeddings

> Generating the embeddings from the verb could be done another way than simply entering the list into the embeddings model

In [19]:
df['Sentence Embeddings'] = df['Example'].apply(GetEmbeddings)
df['Verb Embeddings'] = df['Verb'].apply(GetEmbeddings)
df['Literal Word Sense Embeddings'] = df['Literal Word Sense'].apply(GetEmbeddings)
df['Non-literal Word Sense Embeddings'] = df['Non-literal Word Sense'].apply(GetEmbeddings)
df.head(3)

,FOS,Example,Non-literal Word Sense,Literal Word Sense,Verb,Sentence Embeddings,Verb Embeddings,Literal Word Sense Embeddings,Non-literal Word Sense Embeddings
0,lapas na sa kalendaryo,"Si Ana lapas na sa kalendaryo,",a person over the age of 31,a person over the age of 31,['lapas'],"[-0.07984485, -0.034076605, -0.025813583, -0.0...","[-0.023157373, -0.0048209582, -0.03747844, -0....","[-0.0016924865, 0.03313424, 0.027779346, -0.03...","[-0.0016924865, 0.03313424, 0.027779346, -0.03..."
1,molayag sa kalipay,Ang mga magkauban nagmolayag s,(euphemistic) to have sexual intercourse,(euphemistic) to have sexual intercourse,['molayag'],"[-0.0004858509, -0.025959043, -0.017933264, 0....","[-0.035887502, 0.001659066, -0.014024007, -0.0...","[0.020938013, -0.045462314, 0.003155057, -0.02...","[0.020938013, -0.045462314, 0.003155057, -0.02..."
2,bunal,Gidala niya ang bunal aron map,stick used in beating or whipping an animal or...,stick used in beating or whipping an animal or...,['bunal'],"[-0.002950007, 0.020861559, 0.037009425, -0.01...","[-0.016403744, -0.009215052, -0.031362697, -0....","[0.022656709, -0.01423366, -0.06568678, -0.014...","[0.022656709, -0.01423366, -0.06568678, -0.014..."


### Compute Similarity Scores

In [30]:
df[:2].apply(lambda row: print(ComputeSimilarity(row['Sentence Embeddings'], row['Literal Word Sense Embeddings'])),axis=1)

tensor([[0.2653]])
tensor([[0.1428]])


0    None
1    None
dtype: object

In [32]:
df['Literal Similarity'] = df.apply(lambda row: ComputeSimilarity(row['Sentence Embeddings'], row['Literal Word Sense Embeddings']),axis=1)
df['Non-literal Similarity'] = df.apply(lambda row: ComputeSimilarity(row['Sentence Embeddings'], row['Non-literal Word Sense Embeddings']),axis=1)

In [34]:
df.head(3)

,FOS,Example,Non-literal Word Sense,Literal Word Sense,Verb,Sentence Embeddings,Verb Embeddings,Literal Word Sense Embeddings,Non-literal Word Sense Embeddings,Literal Similarity,Non-literal Similarity
0,lapas na sa kalendaryo,"Si Ana lapas na sa kalendaryo,",a person over the age of 31,a person over the age of 31,['lapas'],"[-0.07984485, -0.034076605, -0.025813583, -0.0...","[-0.023157373, -0.0048209582, -0.03747844, -0....","[-0.0016924865, 0.03313424, 0.027779346, -0.03...","[-0.0016924865, 0.03313424, 0.027779346, -0.03...",[[tensor(0.2653)]],[[tensor(0.2653)]]
1,molayag sa kalipay,Ang mga magkauban nagmolayag s,(euphemistic) to have sexual intercourse,(euphemistic) to have sexual intercourse,['molayag'],"[-0.0004858509, -0.025959043, -0.017933264, 0....","[-0.035887502, 0.001659066, -0.014024007, -0.0...","[0.020938013, -0.045462314, 0.003155057, -0.02...","[0.020938013, -0.045462314, 0.003155057, -0.02...",[[tensor(0.1428)]],[[tensor(0.1428)]]
2,bunal,Gidala niya ang bunal aron map,stick used in beating or whipping an animal or...,stick used in beating or whipping an animal or...,['bunal'],"[-0.002950007, 0.020861559, 0.037009425, -0.01...","[-0.016403744, -0.009215052, -0.031362697, -0....","[0.022656709, -0.01423366, -0.06568678, -0.014...","[0.022656709, -0.01423366, -0.06568678, -0.014...",[[tensor(0.2126)]],[[tensor(0.2126)]]


## PCA-Guided K-means


## Classification Models

# Output